### Setup & configuration

In [ ]:
PROJECT_NAME = "my_project"

In [ ]:
import os
import sys
import importlib
import json
import time
import mlflow
import pandas as pd
# import torch
import importlib
from IPython.display import display, Markdown, JSON, HTML
# append the parent folder to the system path to allow imports from there
sys.path.append('..')

# Debug

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, ConfigDict, Field

import torch
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================================================
# SCHEMAS
# =========================================================

class ReasoningStep(BaseModel):
    model_config = ConfigDict(extra="forbid")

    step_id: int
    step_type: str
    title: str
    explanation: str

    evidence: List[str] = Field(default_factory=list)
    extracted_entities: List[str] = Field(default_factory=list)

    causal_dependency: Optional[str] = None
    temporal_reference: Optional[str] = None

    confidence: float
    validation_status: str


class IncidentSummary(BaseModel):
    model_config = ConfigDict(extra="forbid")

    summary: str
    root_cause: Optional[str] = None
    impacted_systems: List[str] = Field(default_factory=list)
    severity: Optional[str] = None

    reasonings: List[ReasoningStep] = Field(default_factory=list)


# =========================================================
# MODEL
# =========================================================

MODEL_NAME = "../../model/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    trust_remote_code=False,
)

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float16,
    local_files_only=True,
    trust_remote_code=False,
)



# =========================================================
# PROMPT (KEPT EXACTLY AS REQUESTED)
# =========================================================

PROMPT = """
You are an industrial incident reconstruction system.

TASK:
Generate a structured incident reconstruction report.

RULES:
- Preserve chronology
- Preserve telemetry
- Preserve identifiers
- Preserve technician actions
- Preserve evidence
- Preserve causal relationships
- Do NOT hallucinate

WORKLOG:

INC-88421

Packet loss spike in Montreal cluster EAGG-MTL-02.
Latency 12ms → 241ms.
BGP flaps detected.

RT-EAGG-19:
TEMP=97C
FAN_RPM=0
CRC_ERR=11842

FAN-TRAY-2 failure confirmed.

Technician Maria L replaced fan tray.

BGP reset 10.9.0.1
Convergence 96 sec

Recovery after hardware replacement.
"""


In [ ]:
# =========================================================
# OUTLINES WRAPPER (NEW API)
# =========================================================

model = outlines.from_transformers(hf_model, tokenizer)

# =========================================================
# STRUCTURED GENERATION (NEW OUTLINES API)
# =========================================================

result = model(
    PROMPT,
    IncidentSummary,
    max_new_tokens=512,
    temperature=0.2
)


# =========================================================
# OUTPUT
# =========================================================

print("\n" + "=" * 80)
print("STRUCTURED OUTPUT")
print("=" * 80)


# Syntax

In [ ]:
raise Exception("This notebook is for debugging purposes. Please run the cells one by one to inspect the outputs.")

In [ ]:
from pydantic import BaseModel
from enum import Enum

class Rating(Enum):
    poor = 1
    fair = 2
    good = 3
    excellent = 4

class ProductReview(BaseModel):
    rating: Rating
    pros: list[str]
    cons: list[str]
    summary: str

review = model(
    "Review: The XPS 13 has great battery life and a stunning display, but it runs hot and the webcam is poor quality.",
    ProductReview,
    max_new_tokens=200,
)

review = ProductReview.model_validate_json(review)
print(f"Rating: {review.rating.name}")  # "Rating: good"
print(f"Pros: {review.pros}")           # "Pros: ['great battery life', 'stunning display']"
print(f"Summary: {review.summary}")     # "Summary: Good laptop with great display but thermal issues"

In [ ]:
model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto"),
    AutoTokenizer.from_pretrained(MODEL_NAME)
)